In [ ]:
# ============================================================
# Cell 1: Model selector — set MODEL before running
# Options: "tiny" (90M) or "0.5b" (0.5B)
# ============================================================
MODEL = "tiny"  # change to "0.5b" for the larger model

MODEL_IDS = {
    "tiny": "tiiuae/falcon-h1-tiny-90m",
    "0.5b": "tiiuae/falcon-h1-0.5b",
}
MODEL_ID = MODEL_IDS[MODEL]
OUTPUT_DIR = f"falcon-h1-{MODEL}-pii-lora"
MERGED_DIR = f"falcon-h1-{MODEL}-pii-merged"
GGUF_F16 = f"falcon-h1-{MODEL}-pii_f16.gguf"
GGUF_Q5 = f"falcon-h1-{MODEL}-pii_q5_k_m.gguf"

print(f"Model: {MODEL_ID}")
print(f"LoRA output: {OUTPUT_DIR}")
print(f"GGUF final: {GGUF_Q5}")

In [ ]:
# ============================================================
# Cell 2: Install dependencies
# ============================================================
!pip install -qU transformers peft trl datasets bitsandbytes accelerate huggingface_hub

import torch
import os
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

print("Dependencies loaded, torch:", torch.__version__)

In [ ]:
# ============================================================
# Cell 3: Clone + build llama.cpp (for GGUF conversion + quantize)
# ============================================================
import os, sys, subprocess

LLAMA_CPP_DIR = "/content/llama.cpp"

if not os.path.exists(LLAMA_CPP_DIR):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
    !pip install -q llama.cpp/gguf-py/
    %cd llama.cpp
    !mkdir -p build && cd build && cmake .. -DLLAMA_CUDA=ON && cmake --build . --target quantize -j$(nproc)
    %cd /content
else:
    print("llama.cpp already cloned and built")

QUANTIZE_BIN = os.path.join(LLAMA_CPP_DIR, "build", "bin", "quantize")
CONVERT_SCRIPT = os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py")
print(f"quantize binary: {os.path.exists(QUANTIZE_BIN)}")
print(f"convert script: {os.path.exists(CONVERT_SCRIPT)}")

In [ ]:
# ============================================================
# Cell 4: Check GPU
# ============================================================
!nvidia-smi
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram:.1f} GB")

In [ ]:
# ============================================================
# Cell 5: Load the PII masking dataset
# ============================================================
dataset = load_dataset("ai4privacy/pii-masking-200k", split="train")
print(f"Loaded {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")
print()
print("=== Example ===")
print(f"source_text: {dataset[0]['source_text'][:200]}")
print(f"target_text: {dataset[0]['target_text'][:200]}")

In [ ]:
# ============================================================
# Cell 6: Augment with identity-mapping examples (no PII)
# The original dataset has 0 examples where source == target.
# We add ~10% identity examples so the model learns to pass
# through clean text unchanged.
# ============================================================
import random

IDENTITY_FRACTION = 0.10  # 10% of dataset size

# Sample source_texts and reuse them as identity examples
n_identity = int(len(dataset) * IDENTITY_FRACTION)
source_texts = dataset["source_text"]

random.seed(42)
identity_samples = random.choices(source_texts, k=n_identity)

identity_dataset = Dataset.from_list([
    {"source_text": t, "target_text": t}
    for t in identity_samples
])

# Concatenate and shuffle
augmented = concatenate_datasets([dataset, identity_dataset])
augmented = augmented.shuffle(seed=42)

# Train/eval split
splits = augmented.train_test_split(test_size=0.01, seed=42)
train_dataset = splits["train"]
eval_dataset = splits["test"]

print(f"Original: {len(dataset)} examples")
print(f"Augmented: {len(augmented)} examples (+{n_identity} identity)")
print(f"Train: {len(train_dataset)}  Eval: {len(eval_dataset)}")

In [ ]:
# ============================================================
# Cell 7: Format function — maps source_text/target_text to
# Falcon-H1's chat template ( <|im_start|>user/assistant )
# ============================================================
def format_pii(example):
    messages = [
        {"role": "user", "content": example["source_text"]},
        {"role": "assistant", "content": example["target_text"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

# Quick test once tokenizer is loaded
print("Format function ready — will be applied during training")

In [ ]:
# ============================================================
# Cell 8: QLoRA config + load model
# Falcon-H1 constraints:
#   - Exclude conv1d + out_proj from LoRA
#   - Skip out_proj from 4-bit quantization (llm_int8_skip_modules)
# LoRA target modules: in_proj, x_proj, dt_proj
# ============================================================
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["out_proj"],
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading {MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.enable_input_require_grads()
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["in_proj", "x_proj", "dt_proj"],
    modules_to_save=None,  # don't save full modules
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# ============================================================
# Cell 9: Training
# ============================================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    num_train_epochs=1,
    learning_rate=2e-4,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    report_to="none",
    optim="adamw_8bit",
    eval_strategy="steps",
    eval_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=format_pii,
    max_seq_length=2048,
)

trainer.train()

In [ ]:
# ============================================================
# Cell 10: Save LoRA adapter
# ============================================================
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA saved to {OUTPUT_DIR}/")

In [ ]:
# ============================================================
# Cell 11: Merge LoRA into base model
# Load base in fp16, apply LoRA, merge, save full model.
# ============================================================
print("Loading base model in fp16 for merge...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading LoRA adapter...")
merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
print("Merging...")
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}/")

In [ ]:
# ============================================================
# Cell 12: Inference demo — compare input vs expected vs predicted
# ============================================================
import random

# Pick a few eval examples
random.seed(0)
demos = random.sample(range(len(eval_dataset)), 5)

print(f"{'INPUT':<70} {'EXPECTED':<70}")
print("-" * 140)
for idx in demos:
    ex = eval_dataset[idx]
    inp = ex['source_text'][:67]
    exp = ex['target_text'][:67]
    print(f"{inp:<70} {exp:<70}")

In [ ]:
# ============================================================
# Cell 13: Run inference with the merged model
# ============================================================
import torch

merged_model.eval()

def predict(text, max_new=128):
    messages = [{"role": "user", "content": text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=max_new,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

print("Running inference on demo examples...\n")
for idx in demos:
    ex = eval_dataset[idx]
    inp = ex['source_text']
    exp = ex['target_text']
    pred = predict(inp)
    print("=" * 80)
    print(f"INPUT:    {inp}")
    print(f"EXPECTED: {exp}")
    print(f"PREDICTED: {pred}")
    print()

In [ ]:
# ============================================================
# Cell 14: Convert HuggingFace model to GGUF FP16
# ============================================================
!python {CONVERT_SCRIPT} {MERGED_DIR} --outfile {GGUF_F16} --outtype f16
import os
f16_size = os.path.getsize(GGUF_F16) / 1e9
print(f"GGUF FP16: {GGUF_F16} ({f16_size:.2f} GB)")

In [ ]:
# ============================================================
# Cell 15: Quantize to Q5_K_M
# ============================================================
!{QUANTIZE_BIN} {GGUF_F16} {GGUF_Q5} Q5_K_M
q5_size = os.path.getsize(GGUF_Q5) / 1e9
print(f"GGUF Q5_K_M: {GGUF_Q5} ({q5_size:.2f} GB)")

In [ ]:
# ============================================================
# Cell 16: Download GGUFs from Colab
# ============================================================
from google.colab import files
print("Downloading Q5_K_M GGUF...")
files.download(GGUF_Q5)
print("If you also want the FP16 GGUF, uncomment the next line:")
# files.download(GGUF_F16)